## 1 - imports and setup

In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

sys.path.insert(0, os.path.abspath('..'))

import config
from src.evaluation import evaluate_performance
from src.experiment import load_experiment_log, log_run
from src.model import (
    train_catboost_final,
    train_lightgbm_final,
    train_xgboost_final,
)

## 2 - load data and best params

In [2]:
X_train = pd.read_parquet(config.RESULTS_PATH / "X_train_selected.parquet")
X_val   = pd.read_parquet(config.RESULTS_PATH / "X_val_selected.parquet")
X_test  = pd.read_parquet(config.RESULTS_PATH / "X_test_selected.parquet")
y_train = pd.read_parquet(config.RESULTS_PATH / "y_train.parquet").squeeze()
y_val   = pd.read_parquet(config.RESULTS_PATH / "y_val.parquet").squeeze()
y_test  = pd.read_parquet(config.RESULTS_PATH / "y_test.parquet").squeeze()

with open(config.RESULTS_PATH / "best_params_catboost.json")  as f: cat_params = json.load(f)
with open(config.RESULTS_PATH / "best_params_xgboost.json")   as f: xgb_params = json.load(f)
with open(config.RESULTS_PATH / "best_params_lightgbm.json")  as f: lgb_params = json.load(f)

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")
print(f"y_train fraud rate: {y_train.mean():.4f}")
print("\ncatboost params:", cat_params)
print("xgboost params: ", xgb_params)
print("lightgbm params:", lgb_params)

# string cols were label-encoded and originals dropped in pipeline
cat_features = []

X_train: (413378, 115)  X_val: (88581, 115)  X_test: (88581, 115)
y_train fraud rate: 0.0352

catboost params: {'iterations': 1300, 'learning_rate': 0.08822094140606979, 'depth': 8, 'l2_leaf_reg': 7.857800899303629, 'bagging_temperature': 0.17920044268272478}
xgboost params:  {'n_estimators': 700, 'learning_rate': 0.0331739565924497, 'max_depth': 8, 'subsample': 0.641701337349787, 'colsample_bytree': 0.9276269504938003, 'min_child_weight': 10}
lightgbm params: {'num_boost_round': 1500, 'learning_rate': 0.03967957572302009, 'max_depth': 8, 'num_leaves': 27, 'min_child_samples': 75, 'subsample': 0.9560642338982602, 'colsample_bytree': 0.856212053346252}


## 3 - train final catboost

In [3]:
models_dir = config.OUTPUT_PATH / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
cat_path = models_dir / 'catboost_final.pkl'

if cat_path.exists():
    cat_model = joblib.load(cat_path)
    print('loaded catboost_final from disk')
else:
    cat_model = train_catboost_final(
        X_train, y_train, X_val, y_val, cat_features, cat_params
    )
    joblib.dump(cat_model, cat_path)
    print('trained and saved catboost_final')

print(f"best iteration: {cat_model.get_best_iteration()}")

loaded catboost_final from disk
best iteration: 1010


## 4 - train final xgboost

In [4]:
xgb_path = models_dir / 'xgboost_final.pkl'

if xgb_path.exists():
    xgb_model = joblib.load(xgb_path)
    print('loaded xgboost_final from disk')
else:
    xgb_model = train_xgboost_final(
        X_train, y_train, X_val, y_val, xgb_params
    )
    joblib.dump(xgb_model, xgb_path)
    print('trained and saved xgboost_final')

print(f"best iteration: {xgb_model.best_iteration}")

loaded xgboost_final from disk
best iteration: 693


## 5 - train final lightgbm

In [5]:
lgb_path = models_dir / 'lightgbm_final.pkl'

if lgb_path.exists():
    lgb_booster = joblib.load(lgb_path)
    print('loaded lightgbm_final from disk')
else:
    lgb_booster = train_lightgbm_final(
        X_train, y_train, X_val, y_val, lgb_params
    )
    joblib.dump(lgb_booster, lgb_path)
    print('trained and saved lightgbm_final')

# native booster -- use booster.predict(), not predict_proba()
print(f"best iteration: {lgb_booster.best_iteration}")

loaded lightgbm_final from disk
best iteration: 1369


## 6 - ensemble

In [6]:
# val probabilities for each final model
cat_val_proba = cat_model.predict_proba(X_val)[:, 1]
xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]
lgb_val_proba = lgb_booster.predict(X_val)

# val AUC per model -- used as ensemble weights
cat_val_auc = roc_auc_score(y_val, cat_val_proba)
xgb_val_auc = roc_auc_score(y_val, xgb_val_proba)
lgb_val_auc = roc_auc_score(y_val, lgb_val_proba)

raw_weights = np.array([cat_val_auc, xgb_val_auc, lgb_val_auc])
weights = raw_weights / raw_weights.sum()

print(f"catboost  val AUC: {cat_val_auc:.5f}  weight: {weights[0]:.4f}")
print(f"xgboost   val AUC: {xgb_val_auc:.5f}  weight: {weights[1]:.4f}")
print(f"lightgbm  val AUC: {lgb_val_auc:.5f}  weight: {weights[2]:.4f}")

ensemble_val_proba = (
    weights[0] * cat_val_proba
    + weights[1] * xgb_val_proba
    + weights[2] * lgb_val_proba
)
ensemble_val_auc = roc_auc_score(y_val, ensemble_val_proba)
print(f"\nensemble  val AUC: {ensemble_val_auc:.5f}")

catboost  val AUC: 0.91799  weight: 0.3330
xgboost   val AUC: 0.92278  weight: 0.3348
lightgbm  val AUC: 0.91585  weight: 0.3322

ensemble  val AUC: 0.92460


## 7 - evaluate all final models on val set

In [7]:
threshold = 0.5

df_scores = pd.DataFrame({
    'catboost_final':    cat_val_proba,
    'xgboost_final':     xgb_val_proba,
    'lightgbm_final':    lgb_val_proba,
    'ensemble_weighted': ensemble_val_proba,
})

comparison = pd.DataFrame([
    evaluate_performance(df_scores, col, y_val.values, threshold=threshold)
    for col in df_scores.columns
]).sort_values('AUC', ascending=False).reset_index(drop=True)

display(comparison)

,Score,Precision,Recall,F1 Score,Accuracy,PR AUC,AUC,F2 Score,G-Mean,MCC,Gini
0,ensemble_weighted,0.58808,0.52893,0.55694,0.97110,0.58878,0.92460,0.53979,0.55772,0.54286,0.84921
1,xgboost_final,0.40931,0.61867,0.49267,0.95624,0.57679,0.92278,0.56125,0.50322,0.48179,0.84555
2,catboost_final,0.85725,0.37311,0.51993,0.97634,0.58784,0.91799,0.42062,0.56555,0.55664,0.83597
3,lightgbm_final,0.32689,0.67226,0.43988,0.94121,0.55604,0.91585,0.55498,0.46878,0.44285,0.83169


In [8]:
# helper to compute full metric dict from proba array
def _metrics(proba, y, thresh=0.5):
    preds = (proba > thresh).astype(int)
    auc     = roc_auc_score(y, proba)
    pr_auc  = average_precision_score(y, proba)
    prec, rec, f1, _ = precision_recall_fscore_support(y, preds, average='binary')
    return {
        'val_auc':       round(float(auc), 5),
        'val_gini':      round(float(2 * auc - 1), 5),
        'val_pr_auc':    round(float(pr_auc), 5),
        'val_precision': round(float(prec), 5),
        'val_recall':    round(float(rec), 5),
        'val_f1':        round(float(f1), 5),
    }

fixed_meta = {'feature_set': 'selected', 'tuned': True}

log_run('catboost_final',    {**cat_params, **fixed_meta},
        _metrics(cat_val_proba, y_val.values),       config.RESULTS_PATH)
log_run('xgboost_final',     {**xgb_params, **fixed_meta},
        _metrics(xgb_val_proba, y_val.values),       config.RESULTS_PATH)
log_run('lightgbm_final',    {**lgb_params, **fixed_meta},
        _metrics(lgb_val_proba, y_val.values),       config.RESULTS_PATH)
log_run('ensemble_weighted', {'models': 'cat+xgb+lgb', **fixed_meta,
                              'weights': weights.tolist()},
        _metrics(ensemble_val_proba, y_val.values),  config.RESULTS_PATH)

logged: catboost_final  val_auc=0.91799
logged: xgboost_final  val_auc=0.92278
logged: lightgbm_final  val_auc=0.91585
logged: ensemble_weighted  val_auc=0.92460


## 8 - full experiment log

In [9]:
experiment_log = load_experiment_log(config.RESULTS_PATH)
display(experiment_log)

,timestamp,model,params.model_type,params.max_depth,params.class_weight,metrics.val_auc,metrics.val_gini,metrics.val_pr_auc,metrics.val_precision,metrics.val_recall,...,params.n_estimators,params.subsample,params.colsample_bytree,params.min_child_weight,params.num_boost_round,params.num_leaves,params.min_child_samples,params.tuned,params.models,params.weights
0,2026-05-21T17:18:41.867337,xgboost_tuned_20_trials,NaN,8.0,NaN,0.92525,NaN,NaN,NaN,NaN,...,700.0,0.641701,0.927627,10.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-05-21T22:30:33.734148,ensemble_weighted,NaN,NaN,NaN,0.92460,0.84921,0.58878,0.58808,0.52893,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,cat+xgb+lgb,"[0.3330132106304947, 0.33475046200554864, 0.33..."
2,2026-05-22T17:52:06.559840,ensemble_weighted,NaN,NaN,NaN,0.92460,0.84921,0.58878,0.58808,0.52893,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,cat+xgb+lgb,"[0.3330132106304947, 0.33475046200554864, 0.33..."
3,2026-05-21T17:18:41.860176,catboost_tuned_20_trials,NaN,NaN,NaN,0.92330,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-05-21T22:30:33.639428,xgboost_final,NaN,8.0,NaN,0.92278,0.84555,0.57679,0.40931,0.61867,...,700.0,0.641701,0.927627,10.0,NaN,NaN,NaN,True,NaN,NaN
5,2026-05-22T17:52:06.481521,xgboost_final,NaN,8.0,NaN,0.92278,0.84555,0.57679,0.40931,0.61867,...,700.0,0.641701,0.927627,10.0,NaN,NaN,NaN,True,NaN,NaN
6,2026-05-21T13:26:01.475049,xgboost_tuned_20_trials,NaN,7.0,NaN,0.92118,NaN,NaN,NaN,NaN,...,900.0,0.655798,0.716858,4.0,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-05-22T17:52:06.444943,catboost_final,NaN,NaN,NaN,0.91799,0.83597,0.58784,0.85725,0.37311,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN
8,2026-05-21T22:30:33.596329,catboost_final,NaN,NaN,NaN,0.91799,0.83597,0.58784,0.85725,0.37311,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN
9,2026-05-21T17:18:41.871504,lightgbm_tuned_20_trials,NaN,8.0,NaN,0.91585,NaN,NaN,NaN,NaN,...,NaN,0.956064,0.856212,NaN,1500.0,27.0,75.0,NaN,NaN,NaN


**AUC progression across stages**

| stage | model | val AUC |
|---|---|---|
| baseline | decision_tree max_depth=10 | ~0.806 |
| full features | catboost / xgboost / lightgbm | ~0.906-0.915 |
| tuned 20 trials | catboost / xgboost / lightgbm | see log |
| final selected features | catboost / xgboost / lightgbm | see log |
| ensemble weighted | cat + xgb + lgb | see log |

feature selection typically narrows the feature set without hurting AUC. the ensemble usually matches or slightly exceeds the best individual model. gini (2*AUC-1) is the production metric used in most MRM reviews.

## 9 - save ensemble and individual probabilities

In [10]:
# test set probabilities
cat_test_proba = cat_model.predict_proba(X_test)[:, 1]
xgb_test_proba = xgb_model.predict_proba(X_test)[:, 1]
lgb_test_proba = lgb_booster.predict(X_test)

ensemble_test_proba = (
    weights[0] * cat_test_proba
    + weights[1] * xgb_test_proba
    + weights[2] * lgb_test_proba
)

config.RESULTS_PATH.mkdir(parents=True, exist_ok=True)

pd.Series(ensemble_val_proba,  name='ensemble_proba').to_frame().to_parquet(
    config.RESULTS_PATH / "ensemble_val_proba.parquet")
pd.Series(ensemble_test_proba, name='ensemble_proba').to_frame().to_parquet(
    config.RESULTS_PATH / "ensemble_test_proba.parquet")

pd.Series(cat_val_proba,  name='catboost_proba').to_frame().to_parquet(
    config.RESULTS_PATH / "catboost_final_val_proba.parquet")
pd.Series(xgb_val_proba,  name='xgboost_proba').to_frame().to_parquet(
    config.RESULTS_PATH / "xgboost_final_val_proba.parquet")
pd.Series(lgb_val_proba,  name='lightgbm_proba').to_frame().to_parquet(
    config.RESULTS_PATH / "lightgbm_final_val_proba.parquet")

print("saved ensemble_val_proba.parquet")
print("saved ensemble_test_proba.parquet")
print("saved catboost_final_val_proba.parquet")
print("saved xgboost_final_val_proba.parquet")
print("saved lightgbm_final_val_proba.parquet")

saved ensemble_val_proba.parquet
saved ensemble_test_proba.parquet
saved catboost_final_val_proba.parquet
saved xgboost_final_val_proba.parquet
saved lightgbm_final_val_proba.parquet


In [11]:
# train set probabilities for uniform scorer fitting and PSI reference
cat_train_proba = cat_model.predict_proba(X_train)[:, 1]
xgb_train_proba = xgb_model.predict_proba(X_train)[:, 1]
lgb_train_proba = lgb_booster.predict(X_train)

ensemble_train_proba = (
    weights[0] * cat_train_proba
    + weights[1] * xgb_train_proba
    + weights[2] * lgb_train_proba
)

pd.Series(cat_train_proba,      name="catboost_proba").to_frame().to_parquet(
    config.RESULTS_PATH / "catboost_final_train_proba.parquet")
pd.Series(xgb_train_proba,      name="xgboost_proba").to_frame().to_parquet(
    config.RESULTS_PATH / "xgboost_final_train_proba.parquet")
pd.Series(lgb_train_proba,      name="lightgbm_proba").to_frame().to_parquet(
    config.RESULTS_PATH / "lightgbm_final_train_proba.parquet")
pd.Series(ensemble_train_proba, name="ensemble_proba").to_frame().to_parquet(
    config.RESULTS_PATH / "ensemble_train_proba.parquet")

print(f"cat_train_proba shape:      {cat_train_proba.shape}")
print(f"xgb_train_proba shape:      {xgb_train_proba.shape}")
print(f"lgb_train_proba shape:      {lgb_train_proba.shape}")
print(f"ensemble_train_proba shape: {ensemble_train_proba.shape}")
print("saved catboost_final_train_proba.parquet")
print("saved xgboost_final_train_proba.parquet")
print("saved lightgbm_final_train_proba.parquet")
print("saved ensemble_train_proba.parquet")

cat_train_proba shape:      (413378,)
xgb_train_proba shape:      (413378,)
lgb_train_proba shape:      (413378,)
ensemble_train_proba shape: (413378,)
saved catboost_final_train_proba.parquet
saved xgboost_final_train_proba.parquet
saved lightgbm_final_train_proba.parquet
saved ensemble_train_proba.parquet
